# SLO Burn Rate Orchestration & Multi-Window Alerting Analysis
**Date:** 2026-06-18 | **Author:** Inference Systems Engineering Team
**ADR Reference:** [ADR-010](file:///home/btpl-lap-22/live/obs/notebooks/runbooks/decisions/20260618-010-slo-burn-worker.md) — SLO Burn Rate Evaluation and Multi-Window Alerting Worker

---

This notebook provides interactive verification and mathematical validation of the `slo-burn-worker` logic, demonstrating how multi-window burn rates are calculated, how alert severities are determined, and how deduplication works.

## Section 0 — Setup & Path Configuration

In [1]:
import sys
import os

# Ensure packages are resolvable on sys.path
REPO_ROOT = "/home/btpl-lap-22/live/obs"
WORKER_PATH = os.path.join(REPO_ROOT, "packages/python/slo-burn-worker")
SRC_PATH = os.path.join(WORKER_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print("Environment setup completed successfully.")

Environment setup completed successfully.


## Section 1 — Multi-Window Burn-Rate Computations

Let's import `SloService` and compute the burn rates under different scenarios to verify that a burn rate of $1.0$ matches consumption of exactly $100\%$ of the error budget.

In [2]:
from features.slo.service import SloService

# 95% target means budget is 5% errors
slo_threshold = 0.95

# 100 total requests, 5 errors
br_exact = SloService.compute_burn_rate(errors=5, total=100, slo_threshold=slo_threshold)
print(f"Burn Rate when errors = budget: {br_exact}")

# 100 total requests, 10 errors
br_high = SloService.compute_burn_rate(errors=10, total=100, slo_threshold=slo_threshold)
print(f"Burn Rate when errors = 2x budget: {br_high}")

# 100 total requests, 0 errors
br_zero = SloService.compute_burn_rate(errors=0, total=100, slo_threshold=slo_threshold)
print(f"Burn Rate when errors = 0: {br_zero}")

Burn Rate when errors = budget: 1.0
Burn Rate when errors = 2x budget: 2.0
Burn Rate when errors = 0: 0.0


## Section 2 — Severity Escalation Rules

We verify the multi-window alerting threshold logic:
* **Page**: Burn rate $> 14.4$ (5m) AND $> 6.0$ (1h)
* **Slack**: Burn rate $> 6.0$ (1h) AND $> 3.0$ (6h)
* **Ticket**: Burn rate $> 1.0$ (6h)

In [3]:
# Test severities
s1 = SloService.determine_severity(fast=15.0, medium=7.0, slow=4.0)
print(f"Scenario 1 (High Spikes) -> severity: {s1}")

s2 = SloService.determine_severity(fast=5.0, medium=7.0, slow=4.0)
print(f"Scenario 2 (Moderate Degradation) -> severity: {s2}")

s3 = SloService.determine_severity(fast=0.5, medium=0.8, slow=1.5)
print(f"Scenario 3 (Slow Burn) -> severity: {s3}")

s4 = SloService.determine_severity(fast=0.2, medium=0.4, slow=0.8)
print(f"Scenario 4 (Under SLO) -> severity: {s4}")

Scenario 1 (High Spikes) -> severity: page
Scenario 2 (Moderate Degradation) -> severity: slack
Scenario 3 (Slow Burn) -> severity: ticket
Scenario 4 (Under SLO) -> severity: None
